In [9]:
import sys

from models import flash_model
from utils import get_text

sys.path.insert(0,"src")
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from subagents import rag_subagent
from langchain_core.messages import HumanMessage

In [10]:
r = rag_subagent["runnable"].invoke({"messages": [HumanMessage(content="what does the retrieval index module do?")]})

D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampli

In [11]:
print(r["messages"][-1].content)

Based on the provided context, the retrieval index module (defined in `src\retrieval\index.py`) handles building and loading the local vector index that the P2 RAG agent retrieves from [src\retrieval\index.py]. 

Specifically, it performs the following functions:
* **`build_index()`**: Chunks every source document, embeds it, and persists it to disk [src\retrieval\index.py].
* **`get_retriever(k=3)`**: Loads the persisted index and returns a retriever over it [src\retrieval\index.py].


In [12]:
from deepagents import create_deep_agent
from models.llm import lite_model

In [13]:
from prompts.template.builder import build_system_prompt
prompt = build_system_prompt(
    assistant_name="Coordinator",
    role="a coordinator who delegates to specialized subagents",
    domain="this codebase and its indexed corpus",
    mission="Answer questions about the project by delegating to the right subagent.",
    tools=(
        """Use rag_subagent (via the task tool) for ANY question about content in the indexed document corpus (code, docs, notes about this project). Do not use filesystem tools (glob/ls/grep) to answer such questions — always delegate first."""
    ),
    output_format="Return the subagents answer directly.",
    constraints="Always delegate the corpus-content questions to rag_subagent first."

)

In [16]:
from deepagents.backends import FilesystemBackend
from models.llm import flash_model
from pathlib import Path
root_dir = Path.cwd().parents[1]

backend = FilesystemBackend(root_dir=root_dir)
agent = create_deep_agent(
    model=flash_model,
    system_prompt="You are a coordinator with access to specialized tools and skills.",
    subagents=[rag_subagent],
    skills=["/skills/"],
    backend=backend,
)

In [17]:
r2 = agent.invoke(
    {
        "messages": HumanMessage(
            content="what does the retrieval index module do?"
        )
    }
)

D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature, top_p will be ignored.
  request = self._build_request_config(
D:\Repos\lang_claude_exercise\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampli

In [19]:
from pprint import pprint
from utils import get_text
result = r2["messages"][-1]
print(get_text(result))

The **retrieval index module** (`src/retrieval/index.py`) is responsible for building, persisting, and loading the local vector index used by the RAG retrieval system.

### Purpose and Functionality
- **Offline Indexing**: Ingestion and indexing are handled offline (e.g., via `uv run python src/retrieval/index.py`) rather than on the query path, avoiding the overhead of re-embedding the corpus on every question.
- **Vector Retrieval**: Provides access to the persisted vector store so retrieval tools/agents can query indexed document chunks.

---

### Core Components & Configuration
- **Vector Store**: Uses **Chroma** (`langchain_chroma`), persisted locally to the `.chroma_index` directory with the collection name `"repo_corpus"`.
- **Embedding Model**: Powered by Google Generative AI embeddings (`GoogleGenerativeAIEmbeddings`) using the `gemini-embedding-2-preview` model.
- **Text Splitting**: Employs `RecursiveCharacterTextSplitter` configured with a `chunk_size` of `1000` and a `chun

In [21]:
for m in r2["messages"]:
    for tc in getattr(m, "tool_calls", None) or []:
        print(tc)

{'name': 'read_file', 'args': {'limit': 1000, 'file_path': '/skills/retrieval-strategy/SKILL.md'}, 'id': 'call_93344', 'type': 'tool_call'}
{'name': 'task', 'args': {'description': 'What does the retrieval index module do? Please search the corpus and explain its functionality, components, and purpose in detail.', 'subagent_type': 'rag_subagent'}, 'id': 'call_196579', 'type': 'tool_call'}
